In [5]:
pip install scikeras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 796.9 kB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 1.2 MB/s eta 0:00:00m eta 0:00:010:00:01
  Attempting uninstall: threadpoolctl
    Found existing installation: threadpoolctl 2.2.0
    Uninstalling threadpoolctl-2.2.0:
      Successfully uninstalled threadpoolctl-2.2.0
  Attempting uninstall: joblib
    Found existing installation: joblib 1.2.0
    Uninstalling joblib-1.2.0:
      Successfully uninstalled joblib-1.2.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
Note: you may need to restart the kernel to use updated packages.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from time import time
import tensorflow as tf
from tensorflow.keras import Input, layers
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from keras.callbacks import EarlyStopping
from statsmodels.tsa.api import VAR
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from statsmodels.stats.diagnostic import acorr_ljungbox
try:
    from keras.wrappers.scikit_learn import KerasRegressor
except ImportError:
    from scikeras.wrappers import KerasRegressor


In [2]:
# ====================================================================
# 1. DEFINE CNN-LSTM ARCHITECTURE
# ====================================================================
def CNN_LSTM(time_series, lag_param, dropout):
    # Convolutions across LOB levels
    if time_series == 'OF':
        input_tensor = Input(shape=(lag_param, 20, 1))
        # Combine imbalance information across sides for each level of the order book
        layer_x = layers.Conv2D(filters=16, kernel_size=(1,2), strides=(1,2))(input_tensor)
        layer_x = layers.LeakyReLU(alpha=0.01)(layer_x)
        # Combine imbalance information across time for each side and level of the order book
        layer_x = layers.Conv2D(filters=16, kernel_size=(4,1), padding='same')(layer_x)
        layer_x = layers.LeakyReLU(alpha=0.01)(layer_x)
        layer_x = layers.Conv2D(filters=16, kernel_size=(4,1), padding='same')(layer_x)
        layer_x = layers.LeakyReLU(alpha=0.01)(layer_x)
    elif time_series == 'OFI':
        input_tensor = Input(shape=(lag_param, 10, 1))
        # Combine imbalance information across time for each side and level of the order book
        layer_x = layers.Conv2D(filters=16, kernel_size=(4,1), padding='same')(input_tensor)
        layer_x = layers.LeakyReLU(alpha=0.01)(layer_x)
        layer_x = layers.Conv2D(filters=16, kernel_size=(4,1), padding='same')(layer_x)
        layer_x = layers.LeakyReLU(alpha=0.01)(layer_x)
    else:
        raise Exception("'time_series' should be 'OF' or 'OFI'")

    # Combine imbalance information across all levels of the book
    layer_x = layers.Conv2D(filters=16, kernel_size=(1,10))(layer_x)
    layer_x = layers.LeakyReLU(alpha=0.01)(layer_x)

    # Inception Module
    # Tower 1
    tower_1 = layers.Conv2D(filters=32, kernel_size=(1,1), padding='same')(layer_x)
    tower_1 = layers.LeakyReLU(alpha=0.01)(tower_1)
    tower_1 = layers.Conv2D(filters=32, kernel_size=(3,1), padding='same')(tower_1)
    tower_1 = layers.LeakyReLU(alpha=0.01)(tower_1)
    # Tower 2
    tower_2 = layers.Conv2D(filters=32, kernel_size=(1,1), padding='same')(layer_x)
    tower_2 = layers.LeakyReLU(alpha=0.01)(tower_2)
    tower_2 = layers.Conv2D(filters=32, kernel_size=(5,1), padding='same')(tower_2)
    tower_2 = layers.LeakyReLU(alpha=0.01)(tower_2)  
    # Tower 3
    tower_3 = layers.MaxPooling2D(pool_size=(3,1), padding='same', strides=(1,1))(layer_x)
    tower_3 = layers.Conv2D(filters=32, kernel_size=(1,1), padding='same')(tower_3)
    tower_3 = layers.LeakyReLU(alpha=0.01)(tower_3)

    # Concatenation and reshaping
    layer_x = layers.concatenate([tower_1, tower_2, tower_3], axis=-1)
    layer_x = layers.Reshape(target_shape=(lag_param, 96))(layer_x)
    
    # Insert variational dropout layer
    layer_x = layers.Dropout(dropout)(layer_x, training=True)
    
    # LSTM with 64 hidden units
    layer_x = layers.LSTM(units=64)(layer_x)
    
    # Final output layer
    output = layers.Dense(units=3, activation='softmax')(layer_x)
    
    model = Model(input_tensor, output)
    opt = Adam(learning_rate=0.01, epsilon=1)
    model.compile(loss='categorical_crossentropy', optimizer=opt, metrics=['accuracy'])
    
    return model

model_details = {
    'OF':  {'model': None, 'function': CNN_LSTM, 'data': None},
    'OFI': {'model': None, 'function': CNN_LSTM, 'data': None}
}

In [5]:
# ====================================================================
# 2. LOAD & FORMAT LOBSTER DATA (JUN 21ST)
# ====================================================================
# Replace with your actual file names
LOBSTER_OB_PATH = 'MSFT_2012-06-21_34200000_57600000_orderbook_10.csv'
LOBSTER_MSG_PATH = 'MSFT_2012-06-21_34200000_57600000_message_10.csv'

print(f"Loading LOBSTER Data...")

# 1. Load the Orderbook
lob_data = pd.read_csv(LOBSTER_OB_PATH, header=None)
lob_data = lob_data.iloc[:, :40] # Restrict to first 10 levels

columns = []
for i in range(10):
    columns.extend([f'PRICE_ASK_{i}', f'VOLUME_ASK_{i}', f'PRICE_BID_{i}', f'VOLUME_BID_{i}'])
lob_data.columns = columns

# Convert LOBSTER prices (1/10000ths of a dollar) to Dollars
price_cols = [c for c in lob_data.columns if 'PRICE' in c]
lob_data[price_cols] = lob_data[price_cols] / 10000.0
lob_data = lob_data.astype(float)

# 2. Load the Message File
msg_columns = ['Time', 'Event', 'OrderID', 'Size', 'Price', 'Direction']
msg_data = pd.read_csv(LOBSTER_MSG_PATH, header=None, names=msg_columns)
msg_data['Price'] = msg_data['Price'] / 10000.0 # Normalize price here too

# 3. Concatenate them (LOBSTER files are strictly row-aligned)
# This merges the timestamp and event info alongside your orderbook snapshots
lob_data = pd.concat([msg_data, lob_data], axis=1)

if len(lob_data) > 100001:
    lob_data = lob_data.iloc[:100001].copy()

# Labels: +1 if Ask increases, -1 if Bid decreases, 0 for stationary
print("Generating Labels...")
lob_data['LABEL_1TICK'] = np.zeros(len(lob_data))
for i in range(len(lob_data)-1):
    if (lob_data.loc[i+1, 'PRICE_ASK_0'] + lob_data.loc[i+1, 'PRICE_ASK_0']) > 1.00002 * (lob_data.loc[i, 'PRICE_ASK_0'] + lob_data.loc[i, 'PRICE_ASK_0']):
        lob_data.loc[i, 'LABEL_1TICK'] = 1
    elif (lob_data.loc[i+1, 'PRICE_BID_0'] + lob_data.loc[i+1, 'PRICE_BID_0']) < 0.99998 * (lob_data.loc[i, 'PRICE_BID_0'] + lob_data.loc[i, 'PRICE_BID_0']):
        lob_data.loc[i, 'LABEL_1TICK'] = -1
lob_data = lob_data.head(len(lob_data)-1).reset_index(drop=True)

Loading LOBSTER Data...
Generating Labels...


In [7]:
# ====================================================================
# 3. DERIVE ORDER FLOW (OF) AND ORDER FLOW IMBALANCE (OFI)
# ====================================================================
print("Computing Order Flow (OF)...")
of_data = pd.DataFrame(index=lob_data.index)
for i in range(10):
    of_data[f'bOF_{i}'] = 0.0
    of_data[f'aOF_{i}'] = 0.0

for j in range(1, len(lob_data)):
    for i in range(10):
        # Bid Order Flow
        if lob_data.loc[j, f'PRICE_BID_{i}'] > lob_data.loc[j-1, f'PRICE_BID_{i}']:
            of_data.loc[j, f'bOF_{i}'] = lob_data.loc[j, f'VOLUME_BID_{i}']
        elif lob_data.loc[j, f'PRICE_BID_{i}'] < lob_data.loc[j-1, f'PRICE_BID_{i}']:
            of_data.loc[j, f'bOF_{i}'] = -1 * lob_data.loc[j, f'VOLUME_BID_{i}']
        else:
            of_data.loc[j, f'bOF_{i}'] = lob_data.loc[j, f'VOLUME_BID_{i}'] - lob_data.loc[j-1, f'VOLUME_BID_{i}']
            
        # Ask Order Flow
        if lob_data.loc[j, f'PRICE_ASK_{i}'] > lob_data.loc[j-1, f'PRICE_ASK_{i}']:
            of_data.loc[j, f'aOF_{i}'] = -1 * lob_data.loc[j, f'VOLUME_ASK_{i}']
        elif lob_data.loc[j, f'PRICE_ASK_{i}'] < lob_data.loc[j-1, f'PRICE_ASK_{i}']:
            of_data.loc[j, f'aOF_{i}'] = lob_data.loc[j, f'VOLUME_ASK_{i}']
        else:
            of_data.loc[j, f'aOF_{i}'] = lob_data.loc[j, f'VOLUME_ASK_{i}'] - lob_data.loc[j-1, f'VOLUME_ASK_{i}']

of_data = pd.concat([of_data, lob_data[['LABEL_1TICK']]], axis=1)
of_data = of_data.iloc[1:].reset_index(drop=True)
model_details['OF']['data'] = of_data

print("Computing Order Flow Imbalance (OFI)...")
ofi_data = pd.DataFrame()
for i in range(10):
    ofi_data[f'OFI_{i}'] = of_data[f'bOF_{i}'] - of_data[f'aOF_{i}']
ofi_data = pd.concat([ofi_data, of_data[['LABEL_1TICK']]], axis=1)
model_details['OFI']['data'] = ofi_data

Computing Order Flow (OF)...
Computing Order Flow Imbalance (OFI)...


In [8]:
# ====================================================================
# 4. VECTOR AUTOREGRESSIVE PROCESSES (VAR ORDER & STABILITY)
# ====================================================================
print("Estimating VAR lags...")
for time_series in model_details:
    model = VAR(model_details[time_series]['data'].iloc[:, :-1])
    results = model.fit(maxlags=100, ic='aic')
    var_order = results.k_ar
    print(time_series + ' VAR order estimate: ', var_order)
    model_details[time_series]['var_model'] = results
    model_details[time_series]['lag_param'] = var_order

for time_series in model_details:
    if model_details[time_series]['var_model'].is_stable():
        print(time_series + ' VAR process is stable.')
    else:
        print(time_series + ' VAR process is unstable.')

Estimating VAR lags...
OF VAR order estimate:  19
OFI VAR order estimate:  60
OF VAR process is stable.
OFI VAR process is stable.


In [9]:
# ====================================================================
# 5. PREPPING THE DATA (SPLIT, SCALE, AND FORMAT)
# ====================================================================
train_weight = 0.8
model_details['OF']['train'] = of_data.iloc[:int(len(of_data)*train_weight)].copy()
model_details['OF']['test'] = of_data.iloc[int(len(of_data)*train_weight):].copy()

model_details['OFI']['train'] = ofi_data.iloc[:int(len(ofi_data)*train_weight)].copy()
model_details['OFI']['test'] = ofi_data.iloc[int(len(ofi_data)*train_weight):].copy()

print("Standardizing data...")
for time_series in model_details.values():
    for col in time_series['train'].columns[:-1]:
        mu = float(time_series['train'].loc[:, col].mean())
        sigma = float(time_series['train'].loc[:, col].std())
        stdize_input = lambda x: (x - mu) / sigma
        time_series['train'][col] = time_series['train'][col].apply(stdize_input)
        time_series['test'][col] = time_series['test'][col].apply(stdize_input)

def format_data(data, lag, dimension):
    data = data.values
    shape = data.shape
    X = np.zeros((shape[0]-lag, lag, dimension))
    Y = np.zeros((shape[0]-lag, 1))
    for i in range(shape[0]-lag):
        X[i] = data[i:i+lag, :dimension]
        Y[i] = data[i+lag-1, -1:]
    X = X.reshape(X.shape[0], lag, dimension, 1)
    Y += 1 # relabel as 0, 1, 2
    Y = to_categorical(Y.astype(int))
    return X, Y

print("Formatting Tensors...")
for time_series in model_details.values():
    time_series['train_x'], time_series['train_y'] = format_data(
        time_series['train'], time_series['lag_param'], len(time_series['train'].columns)-1)
    time_series['test_x'], time_series['test_y'] = format_data(
        time_series['test'], time_series['lag_param'], len(time_series['test'].columns)-1)

Standardizing data...
Formatting Tensors...


In [10]:
# ====================================================================
# 6. CROSS VALIDATION & DROPOUT TUNING
# ====================================================================
es = EarlyStopping(monitor='loss', mode='min', verbose=1, patience=5, min_delta=3e-5, restore_best_weights=True)
cv_epochs = 100
batch_size = 32

print("Running TimeSeries Cross-Validation for Optimal Dropout...")
with tf.device('/CPU:0'):
    for time_series in model_details:
        param_grid = {'time_series': [time_series],
                      'lag_param': [model_details[time_series]['lag_param']],
                      'dropout': [0.1, 0.2, 0.3, 0.4, 0.5]}
        tscv = TimeSeriesSplit(n_splits=5)
        model = KerasRegressor(build_fn=model_details[time_series]['function'], 
                               epochs=cv_epochs, batch_size=batch_size, verbose=2)
        grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=tscv, n_jobs=1, verbose=1)
        grid_result = grid.fit(model_details[time_series]['train_x'], model_details[time_series]['train_y'], callbacks=[es])
        model_details[time_series]['dropout'] = grid_result.best_params_['dropout']
        print(f"{time_series} optimally selected dropout rate: {model_details[time_series]['dropout']}")

Running TimeSeries Cross-Validation for Optimal Dropout...
Fitting 5 folds for each of 5 candidates, totalling 25 fits


ValueError: Invalid parameter dropout for estimator KerasRegressor.
This issue can likely be resolved by setting this parameter in the KerasRegressor constructor:
`KerasRegressor(dropout=0.1)`
Check the list of available parameters with `estimator.get_params().keys()`

In [ ]:
# ====================================================================
# 7. TRAINING THE FINAL MODELS
# ====================================================================
max_epochs = 250
print("Training final models...")
with tf.device('/CPU:0'):
    for time_series in model_details:
        model = model_details[time_series]['function'](
            time_series, model_details[time_series]['lag_param'], model_details[time_series]['dropout'])
        model.summary()
        model.fit(model_details[time_series]['train_x'], model_details[time_series]['train_y'], 
                  epochs=max_epochs, batch_size=batch_size, callbacks=[es])
        model_details[time_series]['model'] = model
        model.save(f'{time_series}-CNN-LSTM-SAVED.hdf5', overwrite=True)

In [ ]:
# ====================================================================
# 8. EVALUATION & CLASSIFICATION REPORTS
# ====================================================================
num_iterations = 100
print("Evaluating Model and applying MC Variational Dropout...")
with tf.device('/CPU:0'):
    for time_series in model_details:
        model_details[time_series]['preds'] = [
            model_details[time_series]['model'].predict(model_details[time_series]['test_x'], verbose=0) 
            for _ in range(num_iterations)
        ]
        model_details[time_series]['preds_mean'] = np.mean(model_details[time_series]['preds'], axis=0).reshape(model_details[time_series]['test_y'].shape)

for time_series in model_details:
    model_details[time_series]['pred_y'] = np.zeros(model_details[time_series]['test_y'].shape)
    for i in range(len(model_details[time_series]['test_y'])):
        model_details[time_series]['pred_y'][i, np.argmax(model_details[time_series]['preds_mean'][i])] = 1
    print(f"\n--- {time_series} Classification Report ---")
    print(classification_report(model_details[time_series]['test_y'],
                                model_details[time_series]['pred_y'],
                                target_names=['Down', 'Stationary', 'Up']))

# Confusion Matrix Plots
fig, (ax1,ax2) = plt.subplots(1, 2, sharex=True, sharey=True, figsize=(10, 5))
mat_con1 = confusion_matrix(model_details['OF']['test_y'].argmax(axis=1), model_details['OF']['pred_y'].argmax(axis=1))
mat_con2 = confusion_matrix(model_details['OFI']['test_y'].argmax(axis=1), model_details['OFI']['pred_y'].argmax(axis=1))

for ax in zip((ax1,ax2), (mat_con1,mat_con2), ('OF','OFI')):
    ax[0].matshow(ax[1], cmap=plt.cm.Blues, alpha=0.5)
    for m in range(ax[1].shape[0]):
        for n in range(ax[1].shape[1]):
            ax[0].text(x=n, y=m, s=ax[1][m, n], va='center', ha='center', size='xx-large')

    ax[0].set_xlabel('Predictions', fontsize=16)
    ax[0].set_ylabel('True', fontsize=16)
    ax[0].set_xticks(np.arange(3))
    ax[0].set_xticklabels(['Down', 'Stationary', 'Up'])
    ax[0].set_yticks(np.arange(3))
    ax[0].set_yticklabels(['Down', 'Stationary', 'Up'], rotation='vertical', va='center')
    ax[0].set_title(ax[2] + ' Confusion Matrix', fontsize=15)
fig.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# 9. BUILDING THE TRADING SIGNALS
# ====================================================================
print("Running trading simulations...")

# 9.1 Softmax Trading Strategy
alphas = [0.6, 0.7, 0.8]
for time_series in model_details:
    trading_period = lob_data.index[-len(model_details[time_series]['preds_mean']):]
    for a in alphas:
        position = None
        model_details[time_series][f'softmax_returns_{a}'] = np.zeros(len(trading_period))
        pnl = 0
        num_trades = 0
        mu = 0 
        entry_last = 0
        
        for t in trading_period:
            idx = t - trading_period.start
            
            if position == 'SHORT':
                model_details[time_series][f'softmax_returns_{a}'][idx] = mu * (lob_data.PRICE_BID_0[t-1] - lob_data.PRICE_BID_0[t])
            elif position == 'LONG':
                model_details[time_series][f'softmax_returns_{a}'][idx] = mu * (lob_data.PRICE_ASK_0[t] - lob_data.PRICE_ASK_0[t-1])
            
            if position != 'LONG' and np.argmax(model_details[time_series]['preds_mean'][idx]) == 2 and np.max(model_details[time_series]['preds_mean'][idx]) > a:
                if position == None:
                    mu = 0.3 * lob_data.VOLUME_ASK_0[t]
                long_now = lob_data.PRICE_ASK_0[t]
                if position == 'SHORT':
                    pnl += mu * (entry_last - long_now)
                entry_last = long_now
                position = 'LONG'
                num_trades += 1

            elif position != 'SHORT' and np.argmax(model_details[time_series]['preds_mean'][idx]) == 0 and np.max(model_details[time_series]['preds_mean'][idx]) > a:
                if position == None:
                    mu = 0.3 * lob_data.VOLUME_BID_0[t]
                short_now = lob_data.PRICE_BID_0[t]
                if position == 'LONG':
                    pnl += mu * (short_now - entry_last)
                entry_last = short_now
                position = 'SHORT'
                num_trades += 1
                
        if position == 'SHORT':
            long_now = lob_data.PRICE_ASK_0[trading_period.stop-1]
            model_details[time_series][f'softmax_returns_{a}'][-1] = mu * (entry_last - long_now)
            pnl += mu * (entry_last - long_now)
            num_trades += 1
        elif position == 'LONG':
            short_now = lob_data.PRICE_BID_0[trading_period.stop-1]
            model_details[time_series][f'softmax_returns_{a}'][-1] = mu * (short_now - entry_last)
            pnl += mu * (short_now - entry_last)
            num_trades += 1

        model_details[time_series][f'softmax_profits_{a}'] = pnl
        model_details[time_series][f'softmax_profits_normalized_{a}'] = pnl/num_trades if num_trades > 0 else 0

# 9.2 Bayesian Trading Strategy
threshold = 0.7
beta1 = [0.1, 0.2, 0.3]
beta2 = 0.7

for time_series in model_details:
    trading_period = lob_data.index[-len(model_details[time_series]['preds_mean']):]
    for b1 in beta1:
        position = None
        model_details[time_series][f'bayesian_returns_({b1},{beta2})'] = np.zeros(len(trading_period))
        pnl = 0
        num_trades = 0
        size = 0
        mu = 0
        entry_last = 0

        for t in trading_period:
            idx = t - trading_period.start
            entropy = -np.sum(np.max(model_details[time_series]['preds_mean'][idx]) * np.log(np.max(model_details[time_series]['preds_mean'][idx])))
            
            if position == 'SHORT':
                model_details[time_series][f'bayesian_returns_({b1},{beta2})'][idx] = size * (lob_data.PRICE_ASK_0[t-1] - lob_data.PRICE_ASK_0[t])
            elif position == 'LONG':
                model_details[time_series][f'bayesian_returns_({b1},{beta2})'][idx] = size * (lob_data.PRICE_BID_0[t] - lob_data.PRICE_BID_0[t-1])
            
            if position != 'LONG' and np.argmax(model_details[time_series]['preds_mean'][idx]) == 2 and np.max(model_details[time_series]['preds_mean'][idx]) > threshold:
                if position == None:
                    mu = 0.3 * lob_data.VOLUME_ASK_0[t]
                long_now = lob_data.PRICE_ASK_0[t]
                
                if b1 < entropy < beta2:
                    if position == 'SHORT': pnl += size * (entry_last - long_now)
                    entry_last = long_now; position = 'LONG'; size = mu; num_trades += 1
                elif entropy < b1:
                    if position == 'SHORT': pnl += size * (entry_last - long_now)
                    entry_last = long_now; position = 'LONG'; size = 1.5 * mu; num_trades += 1
                else: 
                    entry_last = long_now; position = 'LONG'; size = 0.5 * mu; num_trades += 1
                    
            elif position != 'SHORT' and np.argmax(model_details[time_series]['preds_mean'][idx]) == 0 and np.max(model_details[time_series]['preds_mean'][idx]) > threshold:
                if position == None:
                    mu = 0.3 * lob_data.VOLUME_BID_0[t]
                short_now = lob_data.PRICE_BID_0[t]
                
                if b1 < entropy < beta2:
                    if position == 'LONG': pnl += size * (short_now - entry_last)
                    entry_last = short_now; position = 'SHORT'; size = mu; num_trades += 1
                elif entropy < b1:
                    if position == 'LONG': pnl += size * (short_now - entry_last)
                    entry_last = short_now; position = 'SHORT'; size = 1.5 * mu; num_trades += 1
                else: 
                    entry_last = short_now; position = 'SHORT'; size = 0.5 * mu; num_trades += 1
                
        if position == 'SHORT':
            long_now = lob_data.PRICE_ASK_0[trading_period.stop-1]
            model_details[time_series][f'bayesian_returns_({b1},{beta2})'][-1] = size * (entry_last - long_now)
            pnl += size * (entry_last - long_now)
            num_trades += 1
        elif position == 'LONG':
            short_now = lob_data.PRICE_BID_0[trading_period.stop-1]
            model_details[time_series][f'bayesian_returns_({b1},{beta2})'][-1] = size * (short_now - entry_last)
            pnl += size * (short_now - entry_last)
            num_trades += 1
                     
        model_details[time_series][f'bayesian_profits_({b1},{beta2})'] = pnl
        model_details[time_series][f'bayesian_profits_normalized_({b1},{beta2})'] = pnl/num_trades if num_trades > 0 else 0

In [ ]:
# ====================================================================
# 10. PLOTTING TRADING RESULTS AND DDR
# ====================================================================
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 18))

labels = [f"Softmax({a})" for a in alphas] + [f"Bayesian({b1},{beta2})" for b1 in beta1]
x = np.arange(len(labels))
width = 0.35

# Cumulative Profits
of_profits = [model_details['OF'][f'softmax_profits_{a}'] for a in alphas] + [model_details['OF'][f'bayesian_profits_({b1},{beta2})'] for b1 in beta1]
ofi_profits = [model_details['OFI'][f'softmax_profits_{a}'] for a in alphas] + [model_details['OFI'][f'bayesian_profits_({b1},{beta2})'] for b1 in beta1]

ax1.bar(x - width/2, of_profits, width, label='OF')
ax1.bar(x + width/2, ofi_profits, width, label='OFI') 
ax1.set_ylabel('Profit', size='xx-large')
ax1.set_title('Profit by Strategy and Model', size='xx-large')
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=45, size='large')
ax1.legend(loc='best')

# Normalized Profits
of_profits_norm = [model_details['OF'][f'softmax_profits_normalized_{a}'] for a in alphas] + [model_details['OF'][f'bayesian_profits_normalized_({b1},{beta2})'] for b1 in beta1]
ofi_profits_norm = [model_details['OFI'][f'softmax_profits_normalized_{a}'] for a in alphas] + [model_details['OFI'][f'bayesian_profits_normalized_({b1},{beta2})'] for b1 in beta1]

ax2.bar(x - width/2, of_profits_norm, width, label='OF')
ax2.bar(x + width/2, ofi_profits_norm, width, label='OFI') 
ax2.set_ylabel('Normalized Profit', size='xx-large')
ax2.set_title('Normalized Profit by Strategy and Model', size='xx-large')
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=45, size='large')
ax2.legend(loc='best')

# DDR (Downward Deviation Ratio)
of_ddr = []
for a in alphas:
    ret = model_details['OF'][f'softmax_returns_{a}']
    of_ddr.append(ret.mean() / np.sqrt((ret[ret < 0]**2).mean() + 1e-9))
for b1 in beta1:
    ret = model_details['OF'][f'bayesian_returns_({b1},{beta2})']
    of_ddr.append(ret.mean() / np.sqrt((ret[ret < 0]**2).mean() + 1e-9))

ofi_ddr = []
for a in alphas:
    ret = model_details['OFI'][f'softmax_returns_{a}']
    ofi_ddr.append(ret.mean() / np.sqrt((ret[ret < 0]**2).mean() + 1e-9))
for b1 in beta1:
    ret = model_details['OFI'][f'bayesian_returns_({b1},{beta2})']
    ofi_ddr.append(ret.mean() / np.sqrt((ret[ret < 0]**2).mean() + 1e-9))

ax3.bar(x - width/2, of_ddr, width, label='OF')
ax3.bar(x + width/2, ofi_ddr, width, label='OFI') 
ax3.set_ylabel('DDR', size='xx-large')
ax3.set_title('DDR by Strategy and Model', size='xx-large')
ax3.set_xticks(x)
ax3.set_xticklabels(labels, rotation=45, size='large')
ax3.legend(loc='best')

plt.tight_layout()
plt.show()

In [ ]:
# ====================================================================
# 11. LJUNG-BOX TEST FOR UNDERFITTING
# ====================================================================
print("Executing Ljung-Box test...")
with tf.device('/CPU:0'):
    for time_series in model_details:
        eps = 1e-9
        residuals = -np.sum(model_details[time_series]['test_y'] * np.log(model_details[time_series]['preds_mean'] + eps), axis=1)
        
        try:
            lb_test = acorr_ljungbox(residuals, lags=[model_details[time_series]['lag_param']], return_df=True)
            p_value = lb_test.iloc[0]['lb_pvalue']
        except TypeError:
            lb_test = acorr_ljungbox(residuals, lags=[model_details[time_series]['lag_param']])
            p_value = lb_test[1][0]
            
        if p_value < 0.01:
            print(time_series + ' is underfitting.')
        else:
            print(time_series + ' is not underfitting.')